# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**TODO — fill this in once you have the paper open next to you.** This section can't be
written from a template; it needs two *actual* findings quoted (in your own words — don't
copy sentences) from FlyRank's research paper, plus a real methodology question about each,
in the same constructive spirit as the live session's walkthrough.

For each of your two findings, answer in your own words:
- **What is the finding, in one sentence?**
- **Where does its label come from?** (What was actually measured, and what proxy — if any —
  stands in for it?)
- **Does the validation design support the claim as stated?** (Random or grouped/time split?
  Does the population the finding is drawn from match the population the claim is made about?)
- **Your question, framed constructively** — the way you'd want your own Week-5 notebook
  questioned, not "gotcha" framing.

*(No code needed for this section — it's markdown-only, same as the hint says.)*


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

**Before:** the same Week-5 features and models (Logistic Regression, Random Forest), but on a
**random row split** — no grouping by client. This is the dishonest version: rows from the same
client can land on both sides, so the model can partly learn "this is client X's pattern"
rather than the general one.

**After:** the same features and models on the **grouped-by-`client_id`** split already used in
Week 5. Every client's rows stay entirely on one side.

A true time-aware split isn't available here: the starter CSV is a single trailing-90-day
snapshot with no repeated time axis to split on (that only becomes meaningful with the
warehouse release's `fact_content_daily_performance` table in a later week). So the honest
comparison this week is random-split vs. grouped-split, and the **gap between them is itself
the finding** — per the leakage-and-validating skill: report both, and if you can't explain the
gap, you're not done.

Same features, same two models, same three metrics (precision@20/50/200) and ROC-AUC as
Week 5, so the only thing that changes between "before" and "after" is the split.


In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
pd.set_option("display.width", 120)

# --- make sure we're inside the repo (safe to re-run) ---
if not os.path.exists("flyrank-ml-internship-starter"):
    get_ipython().system('git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git')
if os.path.basename(os.getcwd()) != "flyrank-ml-internship-starter":
    os.chdir("flyrank-ml-internship-starter")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# --- same leakage-free feature set as Week 5 ---
numeric_features = [
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "search_volume", "competition", "cpc",
]
categorical_features = [
    "content_type", "main_intent", "competition_level",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
numeric_features += ["has_keyword_data", "has_word_count"]

X_numeric = df[numeric_features].fillna(0)
X_categorical = pd.get_dummies(df[categorical_features].fillna("unknown"), prefix=categorical_features)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]
groups = df["client_id"]


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


def fit_and_score(X_train, X_test, y_train, y_test):
    log_reg = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])
    log_reg.fit(X_train, y_train)
    lr_scores = log_reg.predict_proba(X_test)[:, 1]

    rf = RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    rf_scores = rf.predict_proba(X_test)[:, 1]

    rows = []
    for name, scores in [("Logistic Regression", lr_scores), ("Random Forest", rf_scores)]:
        row = {"model": name, "roc_auc": roc_auc_score(y_test, scores)}
        for k in (20, 50, 200):
            row[f"precision@{k}"] = precision_at_k(scores, y_test.values, k)
        rows.append(row)
    result = pd.DataFrame(rows).set_index("model")
    result.insert(0, "base_rate(test)", y_test.mean())
    return result


# ---- BEFORE: random row split (no grouping) ----
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
before = fit_and_score(X_train_r, X_test_r, y_train_r, y_test_r)
print("BEFORE - random row split (dishonest: clients can leak across train/test)")
print(before.round(3))

# ---- AFTER: grouped split by client_id (same as Week 5) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

after = fit_and_score(X_train_g, X_test_g, y_train_g, y_test_g)
print("\nAFTER - grouped split by client_id (honest: no client appears on both sides)")
print(after.round(3))

# ---- the gap itself is the finding ----
gap = (before[["roc_auc", "precision@20", "precision@50", "precision@200"]]
       - after[["roc_auc", "precision@20", "precision@50", "precision@200"]])
print("\nGAP (before - after): how much the random split was overstating performance")
print(gap.round(3))


Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 2.43 MiB/s, done.
Resolving deltas: 100% (161/161), done.
BEFORE - random row split (dishonest: clients can leak across train/test)
                     base_rate(test)  roc_auc  precision@20  precision@50  precision@200
model                                                                                   
Logistic Regression            0.542    0.696          0.95          0.86          0.805
Random Forest                  0.542    0.747          0.80          0.90          0.880

AFTER - grouped split by client_id (honest: no client appears on both sides)
                     base_rate(test)  roc_auc  precision@20  precision@50  precision@200
model                               

## 3. Leakage audit

The same hunt as the earlier signal-audit work, run against this week's final feature set,
following the attack checklist from `hunting-leakage-and-validating`:

1. **No label-derived or sibling columns in the features** — checked programmatically below
   (`trend_pct`, `trend_direction`, `is_declining_label`, and the `_last_30d` / `_prev_30d`
   columns that `trend_pct` is computed from) — asserted absent from the feature matrix, not
   just eyeballed.
2. **The confession test** — deliberately add a known-leaky column (`trend_pct`) back in,
   retrain, and watch the score jump. If it doesn't jump toward ~1.0, the test harness itself
   is broken and the "clean" result above can't be trusted either.
3. **No product-flags-as-features** — `provider_used` / `model_used` confirmed absent (dictionary
   marks these "not a model feature"); none of the threshold tier columns (`freshness_tier`,
   `position_tier`, etc.) are *someone else's decision system* — they're transparent bucket
   transforms of raw numeric columns already in the feature set, not a competing system's flags.
4. **Population check** — every row in the 30k starter slice is used, with no filtering on
   anything from the outcome window (no filtering by `trend_pct`, `trend_direction`, or
   `is_declining_label` itself), so there's no population-selection leakage to disclose here.
5. **Top feature importance sanity-checked**, not celebrated — printed below for the honest
   (grouped-split) Random Forest from Section 2.


In [2]:
# 1. programmatic check: none of the banned columns made it into the feature matrix
banned_columns = {
    "trend_pct", "trend_direction", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "provider_used", "model_used", "content_id", "client_id",
}
present = banned_columns & set(X.columns)
print(f"banned columns present in feature matrix: {sorted(present) if present else 'none - PASS'}")

# 2. the confession test: deliberately reintroduce a known-leaky column and watch the score jump
X_leaky = X_train_g.copy()
X_leaky_test = X_test_g.copy()
X_leaky["trend_pct__DELIBERATELY_LEAKY"] = df.loc[X_train_g.index, "trend_pct"].fillna(0)
X_leaky_test["trend_pct__DELIBERATELY_LEAKY"] = df.loc[X_test_g.index, "trend_pct"].fillna(0)

leaky_rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
leaky_rf.fit(X_leaky, y_train_g)
leaky_scores = leaky_rf.predict_proba(X_leaky_test)[:, 1]
leaky_auc = roc_auc_score(y_test_g, leaky_scores)

honest_auc = after.loc["Random Forest", "roc_auc"]
print(f"\nhonest ROC-AUC (no trend_pct):     {honest_auc:.3f}")
print(f"leaky ROC-AUC (trend_pct added back): {leaky_auc:.3f}")
print(
    "confession confirmed - the harness reacts to a known leak"
    if leaky_auc > honest_auc + 0.05
    else "WARNING - adding a known-leaky column barely moved the score; "
         "the test harness itself may not be sensitive enough to trust the clean result above"
)

# 3 & 4. stated in the markdown above - no code needed beyond the column check in (1)

# 5. top feature importance on the honest (grouped-split) Random Forest from Section 2
honest_rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
honest_rf.fit(X_train_g, y_train_g)
top_features = pd.Series(honest_rf.feature_importances_, index=X_train_g.columns).sort_values(ascending=False)
print("\ntop 10 features, honest grouped-split model:")
print(top_features.head(10))
print(
    "\nsanity check: does the #1 feature plausibly relate to decline, or does it look "
    "suspiciously perfect on its own? (fill in your read once this prints for real)"
)


banned columns present in feature matrix: none - PASS

honest ROC-AUC (no trend_pct):     0.602
leaky ROC-AUC (trend_pct added back): 1.000
confession confirmed - the harness reacts to a known leak

top 10 features, honest grouped-split model:
days_with_impressions    0.168307
impressions_90d          0.113001
avg_position             0.098968
content_age_days         0.076209
word_count               0.038667
char_count               0.035289
position_tier_top_3      0.030863
ctr                      0.024099
age_tier_365+            0.023975
clicks_90d               0.019616
dtype: float64

sanity check: does the #1 feature plausibly relate to decline, or does it look suspiciously perfect on its own? (fill in your read once this prints for real)


## 4. Claim rewrite

**TODO — swap in your own actual sentence from Week 5** (the "fill in after running" paragraph
at the end of your `w05_model.ipynb` Section 4). Below is the pattern to follow, with a
placeholder "before" claim in the style overclaiming tends to take:

**Before (overclaiming):** *"This model predicts which pages will decline, so refresh work
should be prioritized by its ranking."*

**After (safe language):** *"In this evaluation, the Random Forest's ranking was [directionally
associated with / showed higher precision than the baseline at] observed decline on a
client-grouped holdout. This is decision-support for refresh prioritization, not a guarantee for
any individual page — the model was not tested on clients outside this dataset."*

What changed: "predicts" → "associated with / observed on a holdout" (a model doesn't know the
future, it describes a measured pattern); "should be prioritized" → "decision-support" (a human
still decides); an explicit scope limit (client-grouped holdout, not deployment) was added
instead of implied.


In [3]:
safe_words = ["observed", "measured", "directional", "decision-support", "associated", "correlat"]

before_claim = "This model predicts which pages will decline, so refresh work should be prioritized by its ranking."
after_claim = (
    "In this evaluation, the Random Forest's ranking was directionally associated with observed "
    "decline on a client-grouped holdout. This is decision-support for refresh prioritization, "
    "not a guarantee for any individual page - the model was not tested on clients outside this dataset."
)

def safe_language_check(claim):
    hits = [w for w in safe_words if w in claim.lower()]
    return hits

print("BEFORE claim:", before_claim)
print("safe-language words found:", safe_language_check(before_claim) or "none")

print("\nAFTER claim:", after_claim)
print("safe-language words found:", safe_language_check(after_claim) or "none")

print(
    "\nReplace before_claim above with your ACTUAL Week-5 sentence, and after_claim with your "
    "real rewrite, then re-run this cell so the check reflects your own words, not the template."
)


BEFORE claim: This model predicts which pages will decline, so refresh work should be prioritized by its ranking.
safe-language words found: none

AFTER claim: In this evaluation, the Random Forest's ranking was directionally associated with observed decline on a client-grouped holdout. This is decision-support for refresh prioritization, not a guarantee for any individual page - the model was not tested on clients outside this dataset.
safe-language words found: ['observed', 'directional', 'decision-support', 'associated']

Replace before_claim above with your ACTUAL Week-5 sentence, and after_claim with your real rewrite, then re-run this cell so the check reflects your own words, not the template.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.